<a href="https://colab.research.google.com/github/minyi-k03/LargeLanguageModel/blob/Project-Based-Learning(PBL)/Gpt_5_Api_New_Parameters_and_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-5 API - GPT-5 New Params and Tools 예제

## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )

## Reference : https://cookbook.openai.com/examples/gpt-5/gpt-5_new_params_and_tools

## GPT-5 pricing : https://platform.openai.com/docs/pricing

In [ ]:
!pip install --upgrade openai

In [ ]:
!pip show openai

## OpenAI API Key 설정

In [ ]:
from openai import OpenAI

# 2. API Key 설정 (따옴표 안에 키를 붙여넣으세요)
OPENAI_KEY = "Input Your Key"

# 3. 클라이언트 객체 생성 (준비 끝)
client = OpenAI(api_key=OPENAI_KEY)

print("OpenAI 라이브러리 설치 및 설정 완료!")

# 1. Verbosity Parameter

In [ ]:
import pandas as pd
from IPython.display import display

question = "소년과 그의 첫 반려견에 대한 시를 써 주세요."

data = []

for verbosity in ["low", "medium", "high"]:
    response = client.responses.create(
        model="gpt-5-mini",
        input=question,
        text={"verbosity": verbosity}
    )
    #print(response)

    # Extract text (NULL-safe)
    output_text = ""
    for item in (response.output or []):
        content_list = getattr(item, "content", None)
        if not content_list:
            continue
        for block in content_list:
            txt = getattr(block, "text", None)
            if txt:
                output_text += txt

    usage = response.usage
    data.append({
        "Verbosity": verbosity,
        "Sample Output": output_text,
        "Output Tokens": usage.output_tokens
    })

# Create DataFrame
df = pd.DataFrame(data)

# Display nicely with centered headers
pd.set_option('display.max_colwidth', None)
styled_df = df.style.set_table_styles(
    [
        {'selector': 'th', 'props': [('text-align', 'center')]},  # Center column headers
        {'selector': 'td', 'props': [('text-align', 'left')]}     # Left-align table cells
    ]
)

display(styled_df)

# 2.3 Using Verbosity for Coding Use Cases

In [ ]:
#Setting Verbosity Parameter "low"
prompt = "100만 개의 난수로 이루어진 배열을 정렬하는 파이썬 프로그램을 출력하시오."

def ask_with_verbosity(verbosity: str, question: str):
    response = client.responses.create(
        model="gpt-5-mini",
        input=question,
        text={
            "verbosity": verbosity
        }
    )

    # Extract text (NULL-safe)
    output_text = ""
    for item in (response.output or []):
        content_list = getattr(item, "content", None)
        if not content_list:
            continue
        for block in content_list:
            txt = getattr(block, "text", None)
            if txt:
                output_text += txt

    # Token usage details
    usage = response.usage

    print("--------------------------------")
    print(f"Verbosity: {verbosity}")
    print("Output:")
    print(output_text)
    print("Tokens => input: {} | output: {}".format(
        usage.input_tokens, usage.output_tokens
    ))


# Example usage:
ask_with_verbosity("low", prompt)

In [ ]:
#Setting Verbosity Parameter "Medium"
ask_with_verbosity("medium", prompt)

In [ ]:
#Setting Verbosity Parameter "High"
ask_with_verbosity("high", prompt)

# 2. Free‑Form Function Calling

In [ ]:
response = client.responses.create(
    model="gpt-5-mini",
    input="strawberry에 들어 있는 'r'의 개수와 같은 반지름을 가진 원의 넓이를 code_exec 도구를 사용하여 계산하시오.",
    text={"format": {"type": "text"}},
    tools=[
        {
            "type": "custom",
            "name": "code_exec",
            "description": "Executes arbitrary python code",
        }
    ]
)
response.output

In [ ]:
print(response.output[1].input)

In [ ]:
from typing import List, Dict
import json

def run_code_exec(src: str) -> str:
    """
    데모용 Python 실행기 (보안 주의: 실제 서비스에선 샌드박스/자원제한/타임아웃 필수)
    """
    ns = {}
    exec(src, {}, ns)  # 데모: __builtins__ 미차단
    out = ns
    return json.dumps(out, ensure_ascii=False, default=str)

def resolve_with_tools(client: OpenAI, response):
    while True:
        tool_calls = [
            it for it in getattr(response, "output", [])
            if getattr(it, "type", None) == "custom_tool_call"
        ]
        if not tool_calls:
            return response  # 더 처리할 툴콜 없음 = 최종 응답

        outs = []
        for tc in tool_calls:
            name = getattr(tc, "name", "")
            code = getattr(tc, "input", "")
            tool_call_id = getattr(tc, "call_id", None) or getattr(tc, "id", None)

            if name == "code_exec":
                output = run_code_exec(code)
            else:
                output = json.dumps({"error": f"unknown tool {name}"}, ensure_ascii=False)

            return output

code_exec_result = resolve_with_tools(client, response)
print(code_exec_result)

# 4. Minimal Reasoning


In [ ]:
#COT를 최소화 하여 추론하여 답변하는 기능
prompt = "리뷰의 감정을 긍정|중립|부정 중 하나로 분류하시오. 한 단어만 반환하시오."


response = client.responses.create(
    model="gpt-5",
    input= [{ 'role': 'developer', 'content': prompt },
            { 'role': 'user', 'content': '그 식당 음식이 정말 훌륭했어요! 모두에게 추천합니다.' }],
    reasoning = {
        "effort": "minimal"
    },
)

# Extract text (NULL-safe)
output_text = ""
for item in (response.output or []):
    content_list = getattr(item, "content", None)
    if not content_list:
        continue
    for block in content_list:
        txt = getattr(block, "text", None)
        if txt:
            output_text += txt

# Token usage details
usage = response.usage

print("--------------------------------")
print("Output:")
print(output_text)

In [ ]:
import time

CASES = [
    # (name, developer_prompt, user_prompt, expected)
    ("Bayes", "한 단어/숫자만 반환하시오.", "유병률 1%, 민감도 99%, 특이도 99%. 양성일 때 감염 확률을 % 없이 정수로 반올림하여 출력.", "50"),
    ("Py default-arg", "한 줄에 숫자들만, 쉼표로 구분.",
     "```\ndef f(x, arr=[]):\n    arr.append(x)\n    return sum(arr)\nprint(f(1))\nprint(f(2))\nprint(f(3, []))\nprint(f(4))\n```",
     "1,3,3,7"),
    ("Discount/VAT/FX", "숫자만 반환하시오(예: 12.34).",
     "59000원에 15% 할인 후 부가세 10%, 환율 1300원/USD, 소수 둘째 자리 반올림.", "42.43"),
    ("Logic-1truth", "A|B|C 중 하나만 출력.",
     "A:'B는 거짓말쟁이다', B:'C는 거짓말쟁이다', C:'A와 B는 거짓말쟁이다'. 정확히 1명만 진실.", "B"),
    ("Date+60d", "YYYY-MM-DD 형식으로만 출력.",
     "2019-12-31에서 60일 후 날짜를 출력.", "2020-02-29"),
    ("Boxes/defects/remainder", "정수만 출력.",
     "24개/상자 × 17상자, 8% 불량(반올림) 반품, 9명에게 균등 분배 시 남는 캔 수.", "6"),
    ("Min number (sum/11/ends7)", "숫자만 출력.",
     "자릿수 합 18, 11의 배수, 끝자리가 7인 네 자리 수 중 가장 작은 수.", "1287"),
    ("Filter+sum", "정수만 출력.",
     "목록: [(A, 재고 3, 12000), (B, 재고 0, 5000), (C, 재고 2, 7000), (D, 재고 0, 8000)] 재고=0 항목 가격 합.", "13000"),
]

def run_case(effort, case):
    name, dev, user, expected = case
    kwargs = dict(model="gpt-5",
                  input=[{"role":"developer","content":dev},
                         {"role":"user","content":user}],
                  # verbosity를 낮춰 길이 통제(선택)
                  text={"verbosity":"low"})
    if effort is not None:
        kwargs["reasoning"] = {"effort": effort}

    t0 = time.perf_counter()
    resp = client.responses.create(**kwargs)
    dt = (time.perf_counter() - t0) * 1000

    out = ""
    for item in (resp.output or []):
        for block in getattr(item, "content", []) or []:
            txt = getattr(block, "text", None)
            if txt: out += txt

    usage = resp.usage
    # 일부 SDK에선 세부 토큰 필드가 다를 수 있으니 방어적으로 처리
    in_tok = getattr(usage, "input_tokens", None)
    out_tok = getattr(usage, "output_tokens", None)
    # reasoning 토큰 세부치가 있으면 함께 로깅
    reason_tok = None
    details = getattr(usage, "output_tokens_details", None)
    if details and isinstance(details, dict):
        reason_tok = details.get("reasoning_tokens")

    ok = (out.strip() == expected)
    return {"name":name, "effort": effort or "default",
            "ok": ok, "answer": out.strip(),
            "ms": round(dt), "in_tokens": in_tok, "out_tokens": out_tok,
            "reasoning_tokens": reason_tok}

# 실행
rows = []
for case in CASES:
    rows.append(run_case(None, case))            # default (medium)
    rows.append(run_case("minimal", case))       # minimal

In [ ]:
from collections import defaultdict
from statistics import mean

def _s(v):
    if v is None: return "-"
    if isinstance(v, bool): return "✓" if v else "✗"
    return str(v)

def _sn(v):
    if v is None: return "-"
    if isinstance(v, (int, float)):
        # 정수면 천 단위 구분, 실수면 소수점 유지
        if isinstance(v, int):
            return f"{v:,}"
        return f"{v:.2f}"
    return str(v)

def detailed_report(rows):
    # 기본 컬럼
    headers = ["#", "name", "effort", "ok", "answer", "ms", "in_toks", "out_toks", "reason_toks"]
    # 일부 행에 expected가 있으면 컬럼 추가 (answer 다음)
    if any("expected" in r for r in rows):
        headers.insert(headers.index("answer") + 1, "expected")

    # 표 데이터 준비
    table = []
    for i, r in enumerate(rows, 1):
        row = {
            "#": i,
            "name": _s(r.get("name")),
            "effort": _s(r.get("effort")),
            "ok": _s(r.get("ok")),
            "answer": _s(r.get("answer")),
            "ms": _sn(r.get("ms")),
            "in_toks": _sn(r.get("in_tokens")),
            "out_toks": _sn(r.get("out_tokens")),
            "reason_toks": _sn(r.get("reasoning_tokens")),
            "expected": _s(r.get("expected")) if "expected" in r else None,
        }
        # headers 순서대로 리스트화
        table.append([row[h] for h in headers])

    # 컬럼 폭 계산
    col_widths = []
    for ci, h in enumerate(headers):
        w = len(h)
        for row in table:
            w = max(w, len(str(row[ci])))
        col_widths.append(w)

    def print_row(vals):
        out = []
        for ci, v in enumerate(vals):
            # 숫자 중심 컬럼 우측 정렬
            if headers[ci] in {"#", "ms", "in_toks", "out_toks", "reason_toks"}:
                out.append(str(v).rjust(col_widths[ci]))
            else:
                out.append(str(v).ljust(col_widths[ci]))
        print("  ".join(out))

    # 헤더 출력
    print("\n=== Detailed Results ===")
    print_row(headers)
    print_row(["-" * w for w in col_widths])
    for row in table:
        print_row(row)

    # 요약 (effort 별 성능/토큰/지연)
    by_effort = defaultdict(list)
    for r in rows:
        by_effort[_s(r.get("effort"))].append(r)

    print("\n=== Summary by effort ===")
    sum_headers = ["effort", "cases", "accuracy(%)", "avg_ms", "avg_in_toks", "avg_out_toks", "avg_reason_toks"]
    print("  ".join(h.ljust(14) for h in sum_headers))
    print("  ".join("-" * 14 for _ in sum_headers))

    for eff, items in by_effort.items():
        n = len(items)
        acc = 100.0 * sum(1 for x in items if x.get("ok")) / n if n else 0.0
        def avg(key):
            vals = [x.get(key) for x in items if isinstance(x.get(key), (int, float))]
            return mean(vals) if vals else 0
        line = [
            eff, str(n), f"{acc:.1f}",
            f"{avg('ms'):.0f}",
            f"{avg('in_tokens'):.0f}",
            f"{avg('out_tokens'):.0f}",
            f"{avg('reasoning_tokens'):.0f}",
        ]
        print("  ".join(s.ljust(14) for s in line))

    # 실패 케이스 상세
    fails = [r for r in rows if r.get("ok") is False]
    if fails:
        print("\n=== Mismatches (ok = ✗) ===")
        for r in fails:
            base = f"- {r.get('name')} [{r.get('effort')}]"
            ans = f"answer='{r.get('answer')}'"
            exp = f", expected='{r.get('expected')}'" if "expected" in r else ""
            print(base + " → " + ans + exp)

# 기존 간단 출력 대신 아래 한 줄로 대체
detailed_report(rows)